In [1]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np
from functools import partial
import joblib

from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from text2graphapi.src.IntegratedSyntacticGraph import ISG

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-16 14:49:52,671; - DEBUG; - Import libraries/modules from :PROD


In [6]:
import seaborn as sns
import matplotlib.pyplot as plt

Define path variables

In [3]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [4]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

SyntaxError: invalid syntax (1504503109.py, line 10)

Connect to databricks for logging results

In [10]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/05 17:54:59 INFO mlflow.tracking.fluent: Autologging successfully enabled for keras.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/05 17:54:59 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.
2025/12/05 17:54:59 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/12/05 17:55:00 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/05 17:55:00 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2025/12/05 17:55:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2025/12/05 17:55:01 WARNING mlflow.tracking.fluent: Exception raised while enabling 

In [ ]:
mlflow.sklearn.autolog(disable=True)
mlflow.xgboost.autolog(disable=True)


What are GPU are the experiments run on

In [11]:
!nvidia-smi

/usr/bin/sh: line 1: nvidia-smi: command not found


In [12]:
running_on_gpu = torch.cuda.is_available()

In [13]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [14]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

69

In [15]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

Classification threshold constant specification

In [13]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [15]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [02:14<00:00, 83.7MB/s]


Successfully loaded 273301 items.


In [16]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [17]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Load validation data

In [18]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:01<00:00, 79.2MB/s] 


Successfully loaded 2500 items.


In [19]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [20]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:10<00:00, 80.4MB/s] 


Successfully loaded 19999 items.


In [21]:
test_data_df = pd.DataFrame(test_data)

# Build graphs and extract features

In [23]:
def build_isg_features(train_df, test_df):
    train_texts1 = train_df["pair"].apply(lambda x: x[0])
    train_texts2 = train_df["pair"].apply(lambda x: x[1])
    test_texts1 = test_df["pair"].apply(lambda x: x[0])
    test_texts2 = test_df["pair"].apply(lambda x: x[1])
    
    print("Converting to graphs \n")
    X_train1 = vectorizer.transform(train_texts1)
    X_train2 = vectorizer.transform(train_texts2)
    X_test1 = vectorizer.transform(test_texts1)
    X_test2 = vectorizer.transform(test_texts2)

    X_train = (X_train1, X_train2)
    X_test = (X_test1, X_test2)
    
    return X_train, X_test

# Set up evaluation functions

Evaluate model function

In [ ]:
def compute_classifier_scores(test_data_df, model):
    start_time = time.time()
    verification_results = []
    model.eval()

    with torch.no_grad():
        for i in tqdm(test_data_df.index, desc="Processing rows"):
            
            resulting_df_row = {}
            resulting_df_row['id']  = test_data_df.loc[i, 'id']
            resulting_df_row['actual_result'] = test_data_df.loc[i, 'same']
            
            text1 = test_data_df.loc[i, 'pair'][0]
            text2 = test_data_df.loc[i, 'pair'][1]

            enc = tokenizer(
                text1,
                text2,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=8192
            ).to(device)

            logits = model(**enc).logits
            probs = F.softmax(logits, dim=-1)
            prob_same = probs[0, 1].item()

            result_df_row["propability_same_author"]
            verification_results.append(resulting_df_row)

    result_df = pd.DataFrame(verification_results)
    print("--- Execution Time: %s seconds ---" % round(time.time() - start_time, 2))

    return result_df

Evaluation function

In [ ]:
def evaluate_results(y_true, y_pred, average='binary'):
    accuracy = round(accuracy_score(y_true, y_pred)*100, 2)
    precision = round(precision_score(y_true, y_pred, average=average)*100, 2)
    recall = round(recall_score(y_true, y_pred, average=average)*100, 2)
    f1 = round(f1_score(y_true, y_pred, average=average)*100, 2)
    return accuracy, precision, recall, f1

Optimal threshold search

In [ ]:
def evaluate_classification_thresholds(result_df, classification_thresholds):
    results = []
    y_embeddings = result_df['propability_same_author']
    y_true = result_df['actual_result']
    for threshold in thresholds:
        y_pred = (result_df["propability_same_author"] >= threshold).astype(int)
        accuracy, precision, recall, f1 = evaluate_results(y_true, y_pred)
        results.append({
            "threshold": threshold,
            "accuracy" : accuracy,
            "precision" : precision,
            "recall" : recall,
            "f1" : f1
        })
    return pd.DataFrame(results)

Create histogram of F1 score for different thresholds

In [ ]:
def plot_f1_vs_threshold(results_df):
    plt.figure(figsize=(9, 5))
    plt.plot(results_df["threshold"], results_df["f1"], linewidth=2)
    plt.xlabel("Threshold")
    plt.ylabel("F1 Score")
    plt.title("F1 Score vs Classification Threshold")
    plt.grid(True)
    plt.tight_layout()
    return plt.gcf()

Create the confusion matrix

In [ ]:
def create_confusion_matrix(y_true, y_pred, labels=[False, True])
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    cm_fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, 
                fmt='d', 
                cmap='Blues',
                xticklabels=labels, 
                yticklabels=labels, 
                ax=ax,
                cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix: {model_id}')
    return cm

Objective function - to train and evaluate all combinations of the model and log the results

In [ ]:
def objective(trial, model_id, train_data_df, test_data_df, classification_thresholds):
    lr = trial.suggest_float("lr", optimizer_learning_rate_range[0], optimizer_learning_rate_range[1], log=True)
    epochs = trial.suggest_int("epochs", epochs_range[0], epochs_range[1])
    batch_size = trial.suggest_categorical("batch_size", batch_size)
    w_decay = trial.suggest_float("weight_decay", weight_decay)
    wu_ratio = trial.suggest_categorical("warmup_ration", warmup_ratio)
    cl_dropout = trial.suggest_float("classifier_dropout", classifier_dropout)

    model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,                # 0 = different author, 1 = same author
    problem_type="single_label_classification" #not multiclass or regression
    )
    config.hidden_dropout = classifier_dropout
    config.attention_dropout = classifier_dropout

    device = torch.device("cuda" if running_on_gpu else "cpu")
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=w_decay)

    trained_model = train_classifier(
        model=model,
        df=train_data_df,
        optimizer=optimizer,
        epochs=epochs,
        batch_size=batch_size,
        warmup_ratio=wu_ratio,
    )

    result_df = compute_classifier_scores(test_data_df, model)
    threshold_results_df = evaluate_classification_thresholds(result_df, thresholds)

    f1_threshold_histogram_fig = plot_f1_vs_threshold(threshold_results_df)
    
    top_row = threshold_results_df.sort_values("f1", ascending=False).iloc[0]
    top_threshold = top_row["propability_same_author"]
    top_f1 = top_row["f1"]
    top_accuracy = top_row["accuracy"]
    top_precision = top_row["precision"]
    top_recall = top_row["recall"]

    y_true = result_df["actual_result"]
    y_pred = (result_df["cosine_similarity"] >= top_threshold).astype(int)
    cm = create_confusion_matrix(y_true, y_pred)

    date_str = datetime.now().strftime("%Y%m%d_%H%M")
    mlflow.set_experiment("/Users/jiripokorny455@gmail.com/JP_authorship_verification_dt")
    with mlflow.start_run(run_name=f"{developer_initials}_{model_name}_results_{classification_type}_{trial.number}"):
        mlflow.log_param("gpu_name", gpu_name)
        mlflow.log_param("gpu_vram_gb", gpu_vram_gb)
        
        mlflow.log_param("model_id ", model_id )
        mlflow.log_param("classification_type", classification_type)
        mlflow.log_param("optimizer_learning_rate", lr)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        
        mlflow.log_param("classification_score_thresholds", classification_thresholds)
        mlflow.log_table(data=result_df, artifact_file="classifier_score_results.json")
        mlflow.log_table(data=threshold_results_df, artifact_file="results_for_different_thresholds.json")

        mlflow.log_metric("top_threshold", top_threshold)
        mlflow.log_metric("top_result_accuracy", top_accuracy)
        mlflow.log_metric("top_precision", top_precision)
        mlflow.log_metric("top_recall", top_recall)
        mlflow.log_metric("top_f1", top_f1)
    
        mlflow.log_figure(f1_threshold_histogram_fig, "f1_results_threshold_histogram.png")
        mlflow.log_figure(cm_fig, "confusion_matrix.png")

    return top_f1   

Log model information + metrics + results table + confusion matrix

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

Save the best model

In [ ]:
torch.save(best_model_state, finetuned_model_weights_path")
model.config.save_pretrained(finetuned_model_path)